Import Libraries

In [3]:
import torch
import torch.nn as nn

GRU Cell

In [5]:
class BasicGRU(nn.Module):
    def __init__(self, input_size, hidden_size):
        super(BasicGRU, self).__init__()

        self.hidden_size = hidden_size

        # Update gate
        self.Wz = nn.Linear(input_size, hidden_size)
        self.Uz = nn.Linear(hidden_size, hidden_size)

        # Reset gate
        self.Wr = nn.Linear(input_size, hidden_size)
        self.Ur = nn.Linear(hidden_size, hidden_size)

        # Candidate hidden state
        self.Wh = nn.Linear(input_size, hidden_size)
        self.Uh = nn.Linear(hidden_size, hidden_size)

        self.sigmoid = nn.Sigmoid()
        self.tanh = nn.Tanh()

    def forward(self, x):

        batch_size, seq_len, _ = x.size()

        h = torch.zeros(batch_size, self.hidden_size).to(x.device)

        outputs = []

        for t in range(seq_len):

            x_t = x[:, t, :]

            # Update gate
            z_t = self.sigmoid(self.Wz(x_t) + self.Uz(h))

            # Reset gate
            r_t = self.sigmoid(self.Wr(x_t) + self.Ur(h))

            # Candidate hidden state
            h_tilde = self.tanh(self.Wh(x_t) + self.Uh(r_t * h))

            # Final hidden state
            h = (1 - z_t) * h + z_t * h_tilde

            outputs.append(h.unsqueeze(1))

        outputs = torch.cat(outputs, dim=1)

        return outputs

In [6]:
input_size = 10
hidden_size = 50
seq_len = 20
batch_size = 32

model = BasicGRU(input_size, hidden_size)

x = torch.randn(batch_size, seq_len, input_size)

output = model(x)

print(output.shape)

torch.Size([32, 20, 50])
